# Notebook 5: Comparison with Sánchez Zaballa (2025)

**Object:** MG3 J163554+3629 (z = 3.648)

This notebook compares our results (from the UltraNest posterior) with the published one-zone SED model from:

> Sánchez Zaballa, J.M. (2025) — SED modelling of MG3 J163554+3629

We compare:
1. Full parameter table — our values vs their values
2. Key derived quantities: Eddington ratio λ_Edd, black hole mass, disk luminosity
3. Posterior distributions overlaid with their point estimates
4. Seed black hole mass implied by each result (cosmological growth constraint)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.table import Table
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Load our UltraNest posterior samples
samples     = np.load('../../results/models/ultranest_samples_1zone.npy')
results_un  = np.load('../../results/models/ultranest_results_1zone.npy', allow_pickle=True).item()
free_params = ['B', 'BulkFactor', 'N', 'M_BH', 'L_Disk', 'T_DT']

print(f'Loaded {len(samples)} posterior samples.')
print(f'logZ (this work) = {results_un["logz"]:.3f} +/- {results_un["logzerr"]:.3f}')

---
## 1. Full parameter table comparison
Our posterior medians and 1σ intervals vs Sánchez Zaballa (2025) values.

In [ ]:
# ============================================================
# OUR RESULTS: posterior medians + 1 sigma
# ============================================================
our_med = np.median(samples, axis=0)
our_lo  = np.percentile(samples, 16, axis=0)
our_hi  = np.percentile(samples, 84, axis=0)

# ============================================================
# SANCHEZ ZABALLA (2025) VALUES
# Update these with the actual published table values
# ============================================================
jose = {
    # Parameter          : (value, uncertainty_or_None)
    'M_BH (M_sun)'       : (1.10e9,   None),
    'L_Disk (erg/s)'     : (1.38e46,  None),
    'B (G)'              : (0.05,     None),
    'BulkFactor'         : (15.0,     None),
    'N (cm-3)'           : (None,     None),  # update if available
    'T_DT (K)'           : (None,     None),  # update if available
    'p'                  : (1.59,     None),  # update if available
    'p_1'                : (3.07,     None),  # update if available
    'gamma_break'        : (400.0,    None),  # update if available
    'accr_eff'           : (0.09,     None),  # update if available
    'theta (deg)'        : (3.0,      None),
    'z'                  : (3.648,    None),
}

# Build comparison table
param_labels = [
    ('B (G)',         0, 'B'),
    ('BulkFactor',    1, 'Gamma'),
    ('N (cm-3)',      2, 'N'),
    ('M_BH (M_sun)',  3, 'M_BH'),
    ('L_Disk (erg/s)',4, 'L_Disk'),
    ('T_DT (K)',      5, 'T_DT'),
]

print(f"{'Parameter':<22} {'This work (median)':<28} {'1sigma range':<28} {'Sanchez Zaballa 2025':<22}")
print('-'*100)
for label, idx, key in param_labels:
    med = our_med[idx]
    lo  = our_lo[idx]
    hi  = our_hi[idx]
    jose_val = jose.get(label, (None, None))[0]
    jose_str = f'{jose_val:.4e}' if jose_val is not None else 'N/A'
    print(f"{label:<22} {med:<28.4e} [{lo:.4e}, {hi:.4e}]          {jose_str}")

# Additional fixed parameters (frozen in our fit)
print('\n--- Fixed parameters (frozen in our UltraNest run) ---')
fixed = [
    ('p',          1.590, jose.get('p',(None,))[0]),
    ('p_1',        3.066, jose.get('p_1',(None,))[0]),
    ('gamma_break',381.0, jose.get('gamma_break',(None,))[0]),
    ('accr_eff',   0.083, jose.get('accr_eff',(None,))[0]),
    ('theta (deg)',3.0,   jose.get('theta (deg)',(None,))[0]),
]
for label, our_val, jose_val in fixed:
    jose_str = f'{jose_val:.4f}' if jose_val is not None else 'N/A'
    print(f"  {label:<20}: ours = {our_val:.4f}   Jose = {jose_str}")

---
## 2. Eddington ratio comparison
λ_Edd = L_Disk / L_Edd where L_Edd = 1.26 × 10³⁸ × (M_BH / M_sun) erg/s

In [ ]:
mbh_idx   = free_params.index('M_BH')
ldisk_idx = free_params.index('L_Disk')

M_BH_s   = samples[:, mbh_idx]
L_disk_s = samples[:, ldisk_idx]
L_Edd_s  = 1.26e38 * M_BH_s
lambda_s = L_disk_s / L_Edd_s

med_lam = np.median(lambda_s)
lo_lam  = np.percentile(lambda_s, 16)
hi_lam  = np.percentile(lambda_s, 84)

# Jose values
MBH_jose    = 1.1e9
Ldisk_jose  = 1.38e46
lambda_jose = Ldisk_jose / (1.26e38 * MBH_jose)

print('=== Eddington ratio comparison ===')
print(f'This work:  lambda_Edd = {med_lam:.4f}  +{hi_lam-med_lam:.4f}  -{med_lam-lo_lam:.4f}')
print(f'Jose (2025): lambda_Edd = {lambda_jose:.4f}')
print(f'Difference: {(lambda_jose - med_lam)/lambda_jose * 100:.1f}% reduction from Jose value')

print('\n=== Black hole mass comparison ===')
med_mbh = np.median(M_BH_s)
lo_mbh  = np.percentile(M_BH_s, 16)
hi_mbh  = np.percentile(M_BH_s, 84)
print(f'This work:  M_BH = {med_mbh:.3e}  +{hi_mbh-med_mbh:.3e}  -{med_mbh-lo_mbh:.3e} M_sun')
print(f'Jose (2025): M_BH = {MBH_jose:.3e} M_sun')

print('\n=== Disk luminosity comparison ===')
med_ld = np.median(L_disk_s)
lo_ld  = np.percentile(L_disk_s, 16)
hi_ld  = np.percentile(L_disk_s, 84)
print(f'This work:  L_Disk = {med_ld:.3e}  +{hi_ld-med_ld:.3e}  -{med_ld-lo_ld:.3e} erg/s')
print(f'Jose (2025): L_Disk = {Ldisk_jose:.3e} erg/s')

---
## 3. Posterior plots overlaid with Jose's values

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

latex_labels = {
    'B':          r'$B$ (G)',
    'BulkFactor': r'$\Gamma$',
    'N':          r'$N$ (cm$^{-3}$)',
    'M_BH':       r'$M_{\rm BH}$ ($M_\odot$)',
    'L_Disk':     r'$L_{\rm disk}$ (erg/s)',
    'T_DT':       r'$T_{\rm DT}$ (K)',
}
jose_vals = {
    'B':          0.05,
    'BulkFactor': 15.0,
    'N':          None,
    'M_BH':       1.1e9,
    'L_Disk':     1.38e46,
    'T_DT':       None,
}

for i, (name, ax) in enumerate(zip(free_params, axes)):
    s = samples[:, i]
    med = np.median(s)
    lo  = np.percentile(s, 16)
    hi  = np.percentile(s, 84)

    ax.hist(s, bins=50, color='steelblue', alpha=0.7, density=True)
    ax.axvline(med, color='navy',  lw=2,   label=f'This work: {med:.3e}')
    ax.axvline(lo,  color='navy',  lw=1.5, ls='--')
    ax.axvline(hi,  color='navy',  lw=1.5, ls='--')
    ax.axvspan(lo, hi, alpha=0.1, color='steelblue')

    jose_v = jose_vals.get(name)
    if jose_v is not None:
        ax.axvline(jose_v, color='red', lw=2, ls=':', label=f'Jose: {jose_v:.3e}')

    ax.set_xlabel(latex_labels[name], fontsize=11)
    ax.set_ylabel('Posterior density', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('MG3 J163554+3629 — Posterior vs Sánchez Zaballa (2025)\n'
             '(blue = this work, red dashed = Jose values)',
             fontsize=12)
plt.tight_layout()
plt.savefig('../../results/plots/comparison_posteriors.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_posteriors.png')

---
## 4. Eddington ratio posterior + Jose reference

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(lambda_s, bins=50, color='steelblue', alpha=0.7,
        density=True, label='This work (UltraNest)')
ax.axvline(med_lam, color='navy', lw=2,
           label=fr'Median = {med_lam:.3f}$^{{+{hi_lam-med_lam:.3f}}}_{{-{med_lam-lo_lam:.3f}}}$')
ax.axvline(lo_lam, color='navy', lw=1.5, ls='--')
ax.axvline(hi_lam, color='navy', lw=1.5, ls='--')
ax.axvspan(lo_lam, hi_lam, alpha=0.15, color='steelblue', label='68% CI')
ax.axvline(lambda_jose, color='red', lw=2, ls=':',
           label=f'Sánchez Zaballa (2025): {lambda_jose:.3f}')
ax.axvline(0.10, color='gray', lw=1, ls='--', alpha=0.5, label='10% Eddington')

ax.set_xlabel(r'$\lambda_{\rm Edd} = L_{\rm disk}/L_{\rm Edd}$', fontsize=13)
ax.set_ylabel('Posterior density', fontsize=12)
ax.set_title('Eddington ratio — MG3 J163554+3629', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('../../results/plots/comparison_lambda_edd.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Seed black hole mass constraint
How massive does the black hole seed need to be to grow to M_BH by z=3.65 (t=1.72 Gyr after z=20)?

M_seed = M_BH × exp(−λ_acc × t_avail / t_Salpeter)

where t_Salpeter(η) = 0.45 × (η/0.1) Gyr

In [ ]:
from scipy.optimize import brentq

t_avail = 1.72  # Gyr (z=20 to z=3.65)
eta     = 0.083  # our best-fit accretion efficiency

def t_salpeter(eta): return 0.45 * (eta / 0.1)
def required_seed(MBH, lambda_acc, eta):
    return MBH * np.exp(-lambda_acc * t_avail / t_salpeter(eta))

lambda_grid = np.linspace(1.0, 20.0, 300)
MBH_this = np.median(M_BH_s)
MBH_jose_v = 1.1e9

seed_this = [required_seed(MBH_this,  l, eta) for l in lambda_grid]
seed_jose = [required_seed(MBH_jose_v, l, eta) for l in lambda_grid]

fig, ax = plt.subplots(figsize=(9, 6))
ax.semilogy(lambda_grid, seed_this, 'steelblue', lw=2.5,
            label=fr'This work ($M_{{BH}}$ = {MBH_this:.2e} $M_\odot$)')
ax.semilogy(lambda_grid, seed_jose, 'red', lw=2, ls='--',
            label=fr'Sánchez Zaballa ($M_{{BH}}$ = {MBH_jose_v:.2e} $M_\odot$)')

# Reference seed mass lines
for mass, label in [(1e2, 'Pop III remnant (~100 M☉)'),
                    (1e4, 'DCBH lower (~10⁴ M☉)'),
                    (1e6, 'DCBH upper (~10⁶ M☉)')]:
    ax.axhline(mass, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax.text(19.3, mass*2, label, ha='right', fontsize=9, color='gray')

ax.axvline(1.0, color='black', lw=1, ls='--', alpha=0.4, label='Eddington (λ=1)')

ax.set_xlabel(r'Accretion rate $\lambda = \dot{M}/\dot{M}_{\rm Edd}$', fontsize=12)
ax.set_ylabel(r'Required seed mass $M_{\rm seed}$ ($M_\odot$)', fontsize=12)
ax.set_title(f'Seed mass constraint at η = {eta}\n'
             f'(t_available = {t_avail} Gyr, z=20 → z=3.65)', fontsize=11)
ax.legend(fontsize=10)
ax.set_xlim(1, 20); ax.set_ylim(1e-6, 1e10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('../../results/plots/comparison_seed_mass.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key values
print('=== Seed mass at Eddington accretion (lambda=1) ===')
print(f'This work:   M_seed = {required_seed(MBH_this,  1.0, eta):.3e} M_sun')
print(f'Jose (2025): M_seed = {required_seed(MBH_jose_v, 1.0, eta):.3e} M_sun')
print(f'\n=== Minimum lambda for DCBH seed (1e4 M_sun) ===')
try:
    l_this = brentq(lambda l: required_seed(MBH_this, l, eta) - 1e4, 1.0, 1000.0)
    l_jose = brentq(lambda l: required_seed(MBH_jose_v, l, eta) - 1e4, 1.0, 1000.0)
    print(f'This work:   lambda_min = {l_this:.2f}')
    print(f'Jose (2025): lambda_min = {l_jose:.2f}')
except Exception as e:
    print(f'Could not compute: {e}')